In [0]:
%sql
CREATE CATALOG IF NOT EXISTS main

In [0]:
%sql
USE CATALOG main;

USE SCHEMA default;

CREATE VOLUME IF NOT EXISTS landing

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS main.bronze;

In [0]:
from pyspark.sql.functions import current_timestamp

caminho_landing = "/Volumes/main/default/landing/"

def ingest_csv_to_bronze(arquivo_csv, nome_tabela):
    """
    Lê um CSV da landing zone, adiciona timestamp de ingestão
    e salva como tabela Delta na camada Bronze.
    """
    print(f"Iniciando processamento: {arquivo_csv} -> main.bronze.{nome_tabela}")
    
    try:
        # Lê o arquivo CSV
        df_csv = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(f"{caminho_landing}{arquivo_csv}")
            
        # Validação de segurança
        if df_csv.count() == 0:
            raise ValueError(f"O arquivo {arquivo_csv} está vazio.")
        
        # Adiciona a coluna exigida com o timestamp exato da inserção
        df_bronze = df_csv.withColumn("timestamp_ingestion", current_timestamp())
        
        # Salva como tabela Delta no banco de dados bronze
        destino = f"main.bronze.{nome_tabela}"
        df_bronze.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(destino)
            
        print(f"Sucesso: Tabela {destino} criada!\n")
        
    except Exception as e:
        print(f"Erro ao processar {arquivo_csv}: {e}\n")

In [0]:
# Mapeamento dos arquivos
mapeamento_arquivos = {
    "olist_customers_dataset.csv": "tb_customers",
    "olist_geolocation_dataset.csv": "tb_geolocalizacao",
    "olist_order_items_dataset.csv": "tb_order_items",
    "olist_order_payments_dataset.csv": "tb_order_payments",
    "olist_order_reviews_dataset.csv": "tb_order_reviews",
    "olist_orders_dataset.csv": "tb_orders",
    "olist_products_dataset.csv": "tb_products",
    "olist_sellers_dataset.csv": "tb_sellers",
    "product_category_name_translation.csv": "tb_product_category_name_translation"
}

# Executa a função para cada item do dicionário
for arquivo, tabela in mapeamento_arquivos.items():
    ingest_csv_to_bronze(arquivo, tabela)

print("Processamento em lote da camada Bronze finalizado!")

In [0]:
import requests
from pyspark.sql.functions import current_timestamp

dbutils.widgets.text("data_inicio", "09-04-2016", "Data Início (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "10-17-2018", "Data Fim (MM-DD-AAAA)")

# Pegar os valores digitados nos widgets
data_inicio_formatada = dbutils.widgets.get("data_inicio")
data_fim_formatada = dbutils.widgets.get("data_fim")

print(f"Buscando cotações de {data_inicio_formatada} até {data_fim_formatada}...")

# Montando a URL
url = f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?@dataInicial='{data_inicio_formatada}'&@dataFinalCotacao='{data_fim_formatada}'&$select=dataHoraCotacao,cotacaoCompra&$format=json"

# Fazendo a chamada para a internet (GET)
resposta = requests.get(url)

# Convertendo a resposta em um dicionário Python (JSON)
dados_json = resposta.json()

# A API do BCB sempre guarda os resultados reais dentro de uma lista chamada 'value'
lista_cotacoes = dados_json.get('value', [])

if lista_cotacoes:
    # Transforma a lista do Python em um DataFrame do Spark
    df_dolar = spark.createDataFrame(lista_cotacoes)
    
    # Adiciona a coluna de timestamp da ingestão (padrão da camada Bronze)
    df_bronze_dolar = df_dolar.withColumn("timestamp_ingestion", current_timestamp())
    
    # Salva como tabela Delta
    destino = "main.bronze.tb_cotacao_dolar"
    
    df_bronze_dolar.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(destino)
        
    print(f"Sucesso! {df_bronze_dolar.count()} registros salvos em {destino}.")
else:
    print("Atenção: A API não retornou dados para este período.")

In [0]:
%sql
SELECT * FROM main.bronze.tb_cotacao_dolar